In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# ================== 0) ورودی‌ها ==================
model_ans = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\qwen3 235\results_few_shot_cot_e2p.csv")  
gold      = pd.read_csv(r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv")  # شامل: idx, gold

model_id_col  = "id"
model_ans_col = "answer"
gold_id_col   = "idx"
gold_ans_col  = "Gold"

# ================== 1) نرمال‌سازی پاسخ‌ها و Gold ==================
def norm_ans(x):
    if pd.isna(x): return None
    s = str(x).strip().strip('"').strip("'")
    try:
        v = int(float(s))
        return str(v) if v in {1,2,3,4} else None
    except:
        for d in ["1","2","3","4"]:
            if s.endswith(d): return d
        return None

def parse_gold_single_or_pair(x):
    """
    Gold ممکن است در یک ردیف (مثلاً idx=52) به صورت '4-2' باشد.
    این تابع همان را به مجموعه پاسخ‌های مجاز تبدیل می‌کند؛ سایر ردیف‌ها تک‌گزینه‌ای هستند.
    """
    if pd.isna(x): 
        return set()
    s = str(x).strip()
    if "-" in s:
        parts = [p.strip() for p in s.split("-")]
        return {norm_ans(p) for p in parts if norm_ans(p) is not None}
    v = norm_ans(s)
    return {v} if v is not None else set()

model = model_ans.copy()
gold2 = gold.copy()

model[model_ans_col] = model[model_ans_col].apply(norm_ans)
gold2["gold_set"] = gold2[gold_ans_col].apply(parse_gold_single_or_pair)
gold2["is_multi"] = gold2["gold_set"].apply(lambda s: len(s) > 1)

# ================== 2) ادغام id↔idx ==================
df = pd.merge(
    model,
    gold2[[gold_id_col, "gold_set", "is_multi"]],
    left_on=model_id_col,
    right_on=gold_id_col,
    how="inner"
)

# صحت (lenient: هر عضو gold_set قابل قبول است)
df["correct"] = df.apply(lambda r: (r[model_ans_col] in r["gold_set"]) if r[model_ans_col] is not None else False, axis=1).astype(int)

# ================== 3) تنظیمات ذخیره خروجی‌ها ==================
root = Path("eval_report")
root.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'Arial'

# ================== 4) eval_accuracy (پوشه اول) ==================
acc_dir = root / "eval_accuracy"
acc_dir.mkdir(exist_ok=True)

acc = df["correct"].mean()*100 if len(df) else 0.0
(acc_dir / "accuracy_overall.txt").write_text(f"Accuracy: {acc:.2f}%\n", encoding="utf-8")

by_pred = df.groupby(model_ans_col)["correct"].agg(["mean","count"]).reset_index()
by_pred["accuracy_%"] = by_pred["mean"]*100
by_pred.drop(columns=["mean"]).to_csv(acc_dir / "accuracy_by_pred.csv", index=False, encoding="utf-8-sig")

conf = pd.crosstab(df[model_ans_col], df["gold_set"].apply(lambda s: sorted(list(s))[0] if len(s)>0 else None),
                   rownames=["pred"], colnames=["gold_primary"], dropna=False).fillna(0).astype(int)
conf.to_csv(acc_dir / "confusion.csv", encoding="utf-8-sig")

df.to_csv(acc_dir / "eval_detailed.csv", index=False, encoding="utf-8-sig")

# نمودارها
plt.figure(figsize=(4,4))
plt.bar(["Accuracy"], [acc], color="#4E79A7"); plt.ylim(0,100)
plt.title("Baseline Accuracy"); plt.ylabel("Accuracy (%)")
plt.text(0, acc+1, f"{acc:.1f}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_overall.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

tmp = by_pred.copy()
tmp[model_ans_col] = tmp[model_ans_col].astype(str)
tmp = tmp.sort_values(model_ans_col)
plt.figure(figsize=(6,4))
plt.bar(tmp[model_ans_col], tmp["accuracy_%"], color="#F28E2B"); plt.ylim(0,100)
plt.xlabel("Predicted option"); plt.ylabel("Accuracy (%)"); plt.title("Accuracy by Predicted Option")
for x,y in zip(tmp[model_ans_col], tmp["accuracy_%"]): plt.text(x, y+1, f"{y:.1f}%", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.savefig(acc_dir / "plot_accuracy_by_pred.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

plt.figure(figsize=(5,4))
sns.heatmap(conf, annot=True, fmt="d", cmap="Blues", cbar=False, linewidths=1, linecolor="white")
plt.title("Confusion Matrix (pred x gold)")
plt.tight_layout(); plt.savefig(acc_dir / "plot_confusion_heatmap.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 5) calibration_confidence (پوشه دوم) ==================
cal_dir = root / "calibration_confidence"
cal_dir.mkdir(exist_ok=True)

if "confidence" in df.columns and df["confidence"].notna().any():
    calib = df.groupby("confidence")["correct"].agg(["mean","count"]).reset_index()
    calib.rename(columns={"mean":"accuracy"}, inplace=True); calib["accuracy_%"] = calib["accuracy"]*100
    calib.to_csv(cal_dir / "calibration_by_conf.csv", index=False, encoding="utf-8-sig")

    # reliability curve
    plt.figure(figsize=(5,4))
    plt.plot(calib["confidence"], calib["accuracy_%"], marker="o", color="#4E79A7", label="Observed")
    plt.plot([1,5],[20,100], linestyle="--", color="gray", label="Ideal (scaled)")  # فقط راهنما
    plt.xticks([1,2,3,4,5]); plt.ylim(0,100)
    plt.xlabel("Confidence"); plt.ylabel("Accuracy (%)"); plt.title("Reliability Curve"); plt.legend()
    plt.tight_layout(); plt.savefig(cal_dir / "plot_reliability_curve.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    # trade-off accuracy vs coverage
    rows=[]
    for t in [1,2,3,4,5]:
        sub = df[df["confidence"]>=t]
        coverage = len(sub)/len(df)*100 if len(df) else 0.0
        acc_t = sub["correct"].mean()*100 if len(sub) else np.nan
        rows.append({"threshold":t,"coverage_%":coverage,"accuracy_%":acc_t,"n":len(sub)})
    pd.DataFrame(rows).to_csv(cal_dir / "threshold_tradeoff.csv", index=False, encoding="utf-8-sig")

# ================== 6) tokens_usage (پوشه سوم) ==================
tok_dir = root / "tokens_usage"
tok_dir.mkdir(exist_ok=True)
tok_cols = [c for c in ["prompt_tokens","completion_tokens","total_tokens"] if c in df.columns]

if tok_cols:
    df[tok_cols].agg(["mean","median","min","max"]).round(2).to_csv(tok_dir / "tokens_summary.csv", encoding="utf-8-sig")
    df.groupby("correct")[tok_cols].mean().round(2).to_csv(tok_dir / "tokens_by_correct.csv", encoding="utf-8-sig")

    if "completion_tokens" in df.columns and df["completion_tokens"].notna().any():
        plt.figure(figsize=(6,4))
        plt.hist(df["completion_tokens"].dropna(), bins=20, color="#59A14F", edgecolor="black")
        plt.xlabel("Completion tokens"); plt.ylabel("Frequency"); plt.title("Completion Tokens Distribution")
        plt.tight_layout(); plt.savefig(tok_dir / "plot_tokens_hist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "completion_tokens" in df.columns and "latency_ms" in df.columns and df["completion_tokens"].notna().any() and df["latency_ms"].notna().any():
        plt.figure(figsize=(6,4))
        colors = df["correct"].map({1:"#59A14F",0:"#E15759"})
        plt.scatter(df["completion_tokens"], df["latency_ms"], c=colors, alpha=0.6, edgecolors="black", linewidths=0.3)
        plt.xlabel("Completion tokens"); plt.ylabel("Latency (ms)"); plt.title("Completion Tokens vs Latency")
        plt.tight_layout(); plt.savefig(tok_dir / "plot_tokens_vs_latency.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 7) explanation_quality (پوشه چهارم) ==================
exp_dir = root / "explanation_quality"
exp_dir.mkdir(exist_ok=True)

if "explain_len_words" in df.columns and df["explain_len_words"].notna().any():
    df.groupby("correct")["explain_len_words"].agg(["mean","median","count"]).round(2).reset_index().to_csv(exp_dir / "explain_len_vs_correct.csv", index=False, encoding="utf-8-sig")

    plt.figure(figsize=(5,4))
    sns.boxplot(data=df, x="correct", y="explain_len_words", palette=["#E15759","#59A14F"])
    plt.xticks([0,1], ["Incorrect","Correct"]); plt.xlabel(""); plt.ylabel("Explanation length (words)")
    plt.title("Explanation Length vs Correctness")
    plt.tight_layout(); plt.savefig(exp_dir / "plot_explain_len_box.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "confidence" in df.columns and df["confidence"].notna().any():
        plt.figure(figsize=(6,4))
        plt.scatter(df["explain_len_words"], df["confidence"], alpha=0.6, color="#AF7AC5", edgecolors="black", linewidths=0.3)
        plt.xlabel("Explanation length (words)"); plt.ylabel("Confidence"); plt.title("Explain Length vs Confidence")
        plt.tight_layout(); plt.savefig(exp_dir / "plot_explain_len_vs_confidence.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 8) latency_profile (پوشه پنجم) ==================
lat_dir = root / "latency_profile"
lat_dir.mkdir(exist_ok=True)

if "latency_ms" in df.columns and df["latency_ms"].notna().any():
    lat_summary = df["latency_ms"].agg(["mean","median","min","max",lambda s: np.percentile(s.dropna(),90), lambda s: np.percentile(s.dropna(),95)]).round(1)
    lat_summary.index = ["mean","median","min","max","p90","p95"]
    lat_summary.to_csv(lat_dir / "latency_summary.csv", header=False, encoding="utf-8-sig")

    plt.figure(figsize=(6,4))
    plt.hist(df["latency_ms"].dropna(), bins=20, color="#4E79A7", edgecolor="black")
    plt.xlabel("Latency (ms)"); plt.ylabel("Frequency"); plt.title("Latency Distribution")
    plt.tight_layout(); plt.savefig(lat_dir / "plot_latency_dist.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

    if "confidence" in df.columns and df["confidence"].notna().any():
        plt.figure(figsize=(6,4))
        plt.scatter(df["confidence"], df["latency_ms"], alpha=0.6, color="#E15759", edgecolors="black", linewidths=0.3)
        plt.xlabel("Confidence"); plt.ylabel("Latency (ms)"); plt.title("Confidence vs Latency")
        plt.tight_layout(); plt.savefig(lat_dir / "plot_conf_vs_latency.png", dpi=200, bbox_inches="tight", facecolor="white"); plt.close()

# ================== 9) error_inspection (پوشه ششم) ==================
err_dir = root / "error_inspection"
err_dir.mkdir(exist_ok=True)

# رایج‌ترین خطاها (pred→gold_primary)
err_pairs = conf.stack().reset_index(name="count").rename(columns={"pred":model_ans_col, "gold_primary":"gold"})
err_pairs = err_pairs[err_pairs[model_ans_col] != err_pairs["gold"]].sort_values("count", ascending=False)
err_pairs.head(20).to_csv(err_dir / "top_mistakes.csv", index=False, encoding="utf-8-sig")

# نمونه‌های مرزی (در صورت وجود confidence)
if "confidence" in df.columns and df["confidence"].notna().any():
    hard_false = df[(df["correct"]==0) & (df["confidence"]>=4)]
    lucky_true = df[(df["correct"]==1) & (df["confidence"]<=2)]
    hard_false.to_csv(err_dir / "review_hard_false_high_conf.csv", index=False, encoding="utf-8-sig")
    lucky_true.to_csv(err_dir / "review_lucky_true_low_conf.csv", index=False, encoding="utf-8-sig")

print(f"✅ All reports saved under: {root.resolve()}")

C:\Users\sazgar\AppData\Local\Temp\ipykernel_4216\1891451358.py:162: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="correct", y="explain_len_words", palette=["#E15759","#59A14F"])


✅ All reports saved under: F:\Thesis\project\1-BaselineTest\Models Responses\few-shot-cot-e2p\qwen3 235\eval_baseline_report\eval_report


In [2]:
nan_rows = df[df[model_ans_col].isna()].copy()

print(f"🔎 Nan preds: {len(nan_rows)} rows")
print(nan_rows[[model_id_col, "gold_set"]].head(10).to_string(index=False))

# لیست شناسه‌هایی که باید دوباره اجرا شوند
ids_to_rerun = nan_rows[model_id_col].tolist()


🔎 Nan preds: 0 rows
Empty DataFrame
Columns: [id, gold_set]
Index: []
